## RAG Agent

#### Importing libs

In [ ]:
from IPython.display import display
from google.genai import types
import time

import os
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent.parent))
from utils.config import CLIENT, MODEL
from utils.load_prompt import load_prompt
from utils.embedding import search_neighborhood_context

### Prompt Engineering

In [ ]:
def research_agent(target_neighborhood: str) -> dict:
    rag_tool = types.FunctionDeclaration.from_callable(client=CLIENT, callable=search_neighborhood_context)
    rag_tool = types.Tool(function_declarations=[rag_tool])
    system_prompt = load_prompt("research_agent_v2")

    chat = CLIENT.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            tools=[rag_tool],
            temperature=0.2,
            tool_config = types.ToolConfig(
                function_calling_config=types.FunctionCallingConfig(
                    mode="AUTO"
                )
            )
        )
    )
    response = chat.send_message(
        f"TARGET NEIGHBORHOOD: {target_neighborhood}"
    )
    total_tokens = getattr(response.usage_metadata, "total_token_count", None)
    while True:
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            break

        for fn_call in function_calls:
            args = dict(fn_call.args)
            print(args)
            tool_result = search_neighborhood_context(**args)
            response = chat.send_message(
                types.Part.from_function_response(
                    name=fn_call.name,
                    response={"result": tool_result}
                )
            )
            total_tokens += getattr(response.usage_metadata, "total_token_count", None)

    return {
        "result": response.text,
        "metadata": total_tokens
    }

#### Testing the prompt

In [5]:
research_agent(target_neighborhood='Vila Mariana')

{'neighborhood': 'Vila Mariana', 'question': 'Quais são as características gerais do bairro Vila Mariana que o tornam interessante para investimento imobiliário?'}


{'result': '## Análise de Investimento Imobiliário: Vila Mariana\n\nPrezado investidor,\n\nÉ com grande satisfação que apresento a você a Vila Mariana, um bairro que se destaca como uma das regiões mais promissoras para investimento imobiliário em São Paulo. Vamos explorar juntos os motivos que tornam este local uma escolha tão estratégica.\n\n### 1. Visão geral do bairro\n\nA Vila Mariana é um bairro tradicional e nobre, estrategicamente posicionado na zona Centro-Sul de São Paulo. Imagine um lugar que harmoniza a tranquilidade de ruas arborizadas e um ambiente familiar com a efervescência de uma vida urbana completa. É exatamente isso que a Vila Mariana oferece. A região é amplamente reconhecida pela sua alta qualidade de vida, uma infraestrutura urbana robusta e um Índice de Desenvolvimento Humano (IDH) elevado, fatores que, por si só, já indicam um ambiente muito favorável para a valorização do seu investimento.\n\n### 2. Principais fatores de valorização\n\nA Vila Mariana possui c